In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

In [2]:
url = "processed.cleveland.data"

# Column names from UCI documentation
columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope", "ca", "thal", "target"
]
data = pd.read_csv(url, names = columns)
data.replace("?", np.nan, inplace = True)
data.dropna(inplace = True)
data.head()

# y = data['target']
# X = data.drop('target', axis=1)

y = data.pop("target")
X = data

y = (y > 0).astype(int)
y

0      0
1      1
2      1
3      0
4      0
      ..
297    1
298    1
299    1
300    1
301    1
Name: target, Length: 297, dtype: int32

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter = 5000))
])

param_grid = [
    {
        'model__penalty' : ['l1'],
        'model__C' : [0.01, 0.1, 1., 10.],
        'model__solver' : ['liblinear'],
        'model__class_weight' : ['balanced']
    },
    {
        'model__penalty' : ['l2'],
        'model__C' : [0.01, 0.1, 1., 10.],
        'model__solver': ['lbfgs'],
        'model__class_weight' : ['balanced']
    },
    {
        'model__penalty' : ['elasticnet'],
        'model__C' : [0.01, 0.1, 1., 10.],
        'model__l1_ratio' : np.linspace(0.1, 0.9, 10),
        'model__solver': ['saga'],
        'model__class_weight' : ['balanced']
    }
]

grid = GridSearchCV(pipe, param_grid, cv = StratifiedKFold(n_splits = 5, shuffle = True), scoring = 'accuracy')
grid.fit(X_train, np.array(y_train).ravel())
print(grid.best_params_)
model = grid.best_estimator_

coef = model.named_steps['model'].coef_
coef = pd.Series(coef.ravel(), index = X.columns)
coef = coef.sort_values(ascending = False)
print(coef)

{'model__C': 0.01, 'model__class_weight': 'balanced', 'model__l1_ratio': 0.1, 'model__penalty': 'elasticnet', 'model__solver': 'saga'}
ca          0.230034
thal        0.224845
exang       0.156458
oldpeak     0.147708
cp          0.132332
sex         0.101655
slope       0.059403
trestbps    0.038737
age         0.027276
restecg     0.016601
chol        0.000896
fbs         0.000000
thalach    -0.142620
dtype: float64


In [5]:
accuracy_score(y_test, model.predict(X_test))

0.9166666666666666